# AI Engineering Interview Prep
## Section: RAG (Retrieval-Augmented Generation)

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 731+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (80 Qs)",
    "18. FastAPI (60 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "21. System Design & Scenario-Based (100 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 🔍 Section 3 — Retrieval-Augmented Generation (RAG)

### Q1. What is RAG, and why is it important?
**A:** RAG = Retrieval-Augmented Generation. Instead of relying only on what the LLM memorised during training, you:
1. Store your documents in a vector database
2. When a query arrives, search for the most relevant document chunks
3. Inject those chunks into the prompt as context
4. LLM answers based on your real, current documents

Why critical: Solves hallucination for domain-specific/recent knowledge. Keeps answers up-to-date without retraining ($0 vs $millions). Provides citations. Far cheaper than fine-tuning for knowledge-heavy tasks.

🏷️ *Nyaya-Pro is entirely RAG-based — Indian legal corpus in FAISS. Every response is grounded in actual BNS/BNSS/Constitution text, not Gemini's training-time legal knowledge.*


### Q2. Explain the architecture of a basic RAG system.
**A:**
**Offline Indexing Pipeline:**
```
Documents → Loader → Chunker → Embedding Model → Vector DB
```
**Online Query Pipeline:**
```
User Query → Embed Query → Vector Search (top-k) → Rerank → 
Build Prompt (system + context + query) → LLM → Response
```

Five stages:
1. **Ingestion** — load documents (PDF, web, DB)
2. **Chunking** — split into ~512-token pieces with overlap
3. **Embedding** — each chunk → dense vector, stored in vector DB
4. **Retrieval** — embed query, find top-k similar chunks
5. **Generation** — LLM reads (context chunks + query) → grounded answer

🏷️ *Nyaya-Pro: FastAPI backend, FAISS vector DB, LangChain orchestration, sentence-transformers embeddings, cross-encoder reranking, Gemini generation.*


### Q3. What are the key components of a RAG pipeline?
**A:**
| Component | Purpose | Examples |
|-----------|---------|---------|
| Document Loader | Read source files | PyPDF, BeautifulSoup, Unstructured |
| Text Splitter | Chunk documents | RecursiveCharacterTextSplitter, SemanticChunker |
| Embedding Model | Text → vector | sentence-transformers, OpenAI ada-002 |
| Vector Store | Store & search vectors | FAISS, Pinecone, Chroma, Qdrant |
| Retriever | Query the vector store | similarity search, MMR, hybrid |
| Reranker | Reorder retrieved chunks | CrossEncoder, Cohere Rerank |
| LLM | Generate answer from context | Gemini, GPT-4o, Llama |
| Output Parser | Validate & structure response | Pydantic, LangChain parsers |


### Q4. What are chunking strategies, and how do you choose the right chunk size?
**A:**
**Strategies:**
- **Fixed-size**: Split every N characters/tokens. Simple, fast. Problem: cuts sentences mid-thought.
- **Recursive character**: Split on paragraph → sentence → word hierarchy. Preserves natural boundaries. LangChain default.
- **Semantic**: Split where embedding similarity drops (topic change). Best quality, expensive.
- **Sentence-level**: Split by sentences using NLTK/spaCy. Good for precise retrieval.

**Choosing chunk size:**
- Small (128-256 tokens): High precision retrieval, low context for LLM
- Medium (512 tokens): Best balance — use this as your default ✓
- Large (1024+ tokens): Rich context, noisier retrieval

**Always add overlap** (50-100 tokens) to avoid losing cross-boundary information.

🏷️ *Nyaya-Pro: 512 tokens, 50-token overlap, recursive splitting on paragraph breaks — legal sections are naturally paragraph-structured.*


### Q5. Compare fixed-size chunking, semantic chunking, and recursive chunking.
**A:**
| | Fixed-size | Recursive | Semantic |
|--|------------|-----------|---------|
| **Quality** | Low | Medium | High |
| **Speed** | Fast | Fast | Slow |
| **Cost** | Minimal | Minimal | High (embed every sentence) |
| **Natural boundaries** | ❌ | ✅ | ✅✅ |
| **Best for** | Quick prototypes | Production default | High-quality specialised apps |

**Recursive wins** as the production default: it tries paragraph breaks first, then sentence breaks, then word breaks — preserving natural language boundaries at minimal extra cost. Use semantic chunking only if retrieval quality is business-critical and budget allows.


### Q6. What are embedding models, and how do they convert text to vectors?
**A:** Embedding models are neural networks that map text to dense vectors such that semantically similar texts end up close together in vector space.

How they work:
1. Tokenise the input text
2. Pass through a Transformer encoder (BERT-style, bidirectional)
3. Pool the output token embeddings (mean pooling or [CLS] token) → single vector
4. Optionally normalise to unit length

The model is trained on large text corpora using contrastive learning: similar texts (from the same paragraph, positive pairs) are pushed close; dissimilar texts (random negatives) are pushed apart.

Output: a vector of 384-1536 floats that represents the semantic meaning of the text.


### Q7. How do you choose an embedding model for your RAG system?
**A:**
**Key factors:**
1. **Quality** — check MTEB leaderboard (mteb.io) for your task type (retrieval, semantic similarity)
2. **Dimension** — 384-dim (faster, smaller) vs 1536-dim (better quality, more memory)
3. **Context length** — how many tokens can it embed? Legal/medical docs may need 512+ token support
4. **Speed** — how many embeddings/second? Affects indexing time and query latency
5. **Cost** — self-hosted (free compute) vs API (per token)
6. **Multilingual** — do you need to embed non-English text?

**Common choices:**
- `all-MiniLM-L6-v2` (384-dim) — fast, free, good baseline
- `all-mpnet-base-v2` (768-dim) — better quality, still free
- `text-embedding-3-small` (OpenAI) — good quality, low cost, API
- `paraphrase-multilingual-MiniLM-L12-v2` — if you need multilingual support


### Q8. Explain Agentic RAG.
**A:** Standard RAG: query → single retrieval → answer. One-shot, static.

Agentic RAG: the retrieval process is orchestrated by an agent that can:
- **Route** — classify the query and decide which knowledge base to search
- **Multi-hop retrieve** — retrieve → read → decide if more info needed → retrieve again with a new query
- **Self-RAG** — the model generates a retrieval decision token before each sentence ("should I retrieve here?")
- **Iterative refinement** — if first retrieval didn't answer the question, reformulate and retry

When to use: multi-hop questions ("What is the bail provision for murder under BNS compared to old IPC Section 302?"), questions that span multiple domains, research-style queries.

🏷️ *Nyaya-Pro's query routing is Agentic RAG — before retrieval, an LLM call classifies: Constitutional / Criminal / Procedural / Civil, then routes to the right FAISS index subset.*


### Q9. What is hybrid search, and why is it better than pure vector search?
**A:** Hybrid search = **dense vector search** (semantic meaning) + **sparse keyword search** (BM25 / TF-IDF).

Why each alone is insufficient:
- Pure vector: misses exact matches for proper nouns, IDs, specific jargon ("Section 302 BNS")
- Pure keyword: misses semantic matches, synonyms, paraphrases

Combination using Reciprocal Rank Fusion (RRF):
```
RRF_score(doc) = Σ 1/(k + rank_in_system_m)   for each retrieval system m
```
Documents ranking well in BOTH systems score highest.

🏷️ *Nyaya-Pro v2 added BM25 alongside FAISS because pure vector search sometimes missed specific section references like "Section 41A BNSS" — hybrid search nailed exact citation retrieval.*


### Q10. What is re-ranking, and how does it improve RAG retrieval quality?
**A:** Vector search retrieves the top-k chunks quickly but imprecisely. Re-ranking runs a more accurate (but slower) **cross-encoder** model that reads the query AND each chunk together — seeing full token-level interaction.

Two-stage pipeline:
1. **Stage 1 (fast)**: Vector search → top-20 candidates
2. **Stage 2 (accurate)**: Cross-encoder scores all 20 against query → return top-3

Cross-encoder is more accurate because it sees both query and document in the same forward pass (full cross-attention), rather than comparing independent embeddings.

Models: `cross-encoder/ms-marco-MiniLM-L-6-v2`, Cohere Rerank API.

🏷️ *Nyaya-Pro's core accuracy feature: after FAISS retrieval (top-10), cross-encoder reranking returns top-3 → dramatically reduced off-topic legal answers.*


### Q11. How do you handle multi-document and multi-hop questions in RAG?
**A:** Multi-hop questions require combining facts from multiple documents: "What is the bail provision for murder (BNS) and how does it compare to the old IPC provision?"

Approaches:
1. **Iterative retrieval** — retrieve for first sub-question, read, retrieve for second sub-question, synthesise
2. **Query decomposition** — split the compound question into sub-questions, retrieve for each, merge answers
3. **GraphRAG** — build a knowledge graph from documents; traverse relationships to gather connected facts
4. **Augmented generation loop** — generate intermediate reasoning, use it to formulate the next retrieval query
5. **Parent-child chunking** — child chunks for precise retrieval, parent chunks for richer context that may contain both facts

LangGraph or LlamaIndex's SubQuestionQueryEngine handle this automatically.


### Q12. What is the "lost in the middle" problem in RAG systems?
**A:** LLMs recall information better from the beginning and end of their context window (primacy + recency bias). Content buried in the middle of a long context is underutilised.

RAG impact: If you retrieve 20 chunks and dump them all in, chunks 5-15 are ignored.

Fixes:
1. **Retrieve fewer, better chunks** — top-3 to 5 via reranking, not top-20
2. **Position the most relevant chunk first or last**
3. **Context compression** — extract only the key sentences from each chunk before inserting
4. **Use long-context-optimised models** — Gemini 1.5 and Claude 3 are specifically better at using middle context
5. **Structured context** — format retrieved context with clear headings so the model can navigate it


### Q13. How do you evaluate a RAG system? Explain faithfulness, relevance, and context precision/recall.
**A:**
**Retrieval metrics:**
- **Context Precision**: Of the retrieved chunks, what % were actually relevant? (precision)
- **Context Recall**: Of all relevant chunks in the knowledge base, what % did we retrieve? (recall)

**Generation metrics:**
- **Faithfulness**: Is every claim in the answer supported by the retrieved context? Measured by NLI or LLM-judge.
- **Answer Relevance**: Does the answer actually address what the user asked? (not off-topic)

**End-to-end:**
- **Answer Correctness**: Does the answer match the ground truth? (requires a golden test set)

Tool: **RAGAS framework** automates all these metrics using an LLM-as-evaluator.

🏷️ *I use RAGAS on Nyaya-Pro's test set of 50 legal Q&A pairs monthly. Faithfulness is the most critical metric — a legal product cannot hallucinate.*


### Q14. Explain Self-RAG. How does the model decide when to retrieve?
**A:** Self-RAG (Asai et al., 2023) fine-tunes the LLM to generate special reflection tokens that control retrieval:

- `[Retrieve]` → model decides retrieval is needed for this segment
- `[No Retrieve]` → model answers from its own knowledge (no retrieval)
- `[Relevant]` / `[Irrelevant]` → model evaluates whether retrieved chunks are useful
- `[Supported]` / `[Partially Supported]` / `[No Support]` → model self-grades its own output

This allows on-demand, adaptive retrieval — the model retrieves only when needed, not for every query. Avoids unnecessary retrieval for questions the model already knows well.

Trade-off: Requires fine-tuning the base model on Self-RAG training data. More complex than standard RAG but more efficient and higher quality.


### Q15. What is GraphRAG, and when would you use it?
**A:** GraphRAG (Microsoft Research, 2024) first extracts a knowledge graph from documents (entities, relationships, community summaries), then uses this graph structure during retrieval.

Two modes:
- **Local search**: Query-focused retrieval from the graph near relevant entities
- **Global search**: Query against community summaries to get broad cross-document answers

When to use GraphRAG over standard RAG:
- Complex documents with many interconnected entities (research papers, company knowledge)
- Questions requiring connecting dots across many documents ("What are all the connections between entity X and topic Y?")
- Multi-hop reasoning questions that standard vector search can't handle

When NOT to use: Simple factual Q&A, small knowledge bases, cost-sensitive apps (graph construction is expensive).

🏷️ *For Nyaya-Pro V3, I'm evaluating GraphRAG to handle cross-references between BNS and BNSS (criminal law is full of inter-section references).*


### Q16. How do you handle structured data (tables, SQL databases) in a RAG pipeline?
**A:**
**For tables:**
1. **NL2SQL** — convert natural language query to SQL using LLM, execute against DB, format result
2. **Table serialisation** — convert table to text ("Column1: val, Column2: val...") and embed alongside text chunks
3. **Table QA models** — specialised models (TaPaS) for table-based question answering
4. **Metadata extraction** — extract table headers + stats as text, embed those

**For SQL databases:**
1. **LangChain SQLDatabase** — give LLM access to DB schema, let it generate SQL queries
2. **LlamaIndex StructuredDataIndex** — auto-generates SQL from natural language
3. **Hybrid approach** — structured DB for exact lookups, vector DB for semantic search, merge results

**For PDFs with tables:**
- Tools: Camelot, pdfplumber (for digitally created PDFs)
- For scanned: Azure Document Intelligence, AWS Textract


### Q17. What are the common failure modes of RAG systems, and how do you debug them?
**A:**
| Failure Mode | Symptom | Diagnosis | Fix |
|---|---|---|---|
| Wrong chunks retrieved | Answer about wrong topic | Print retrieved chunks — are they relevant? | Better embedding model; hybrid search; reranking |
| Missing chunks | "I don't have information on that" (but info exists) | Check if chunk exists in DB; check retrieval threshold | Lower similarity threshold; check chunking strategy |
| Hallucination despite right context | Wrong answer with right context | Is instruction clear? Is context too noisy? | Tighter system prompt; fewer chunks; lower temp |
| Lost in the middle | Ignores middle chunks | Confirm retrieval is correct; test with position variation | Rerank; reduce chunk count; compress context |
| Chunk boundary issues | Answer misses cross-boundary facts | Inspect chunk splits at boundary | Increase overlap; larger chunks |
| Outdated answers | Correct historically, wrong now | Check document freshness | Re-indexing pipeline; metadata date filtering |


### Q18. How do you handle document updates and maintain freshness in a RAG system?
**A:**
1. **Soft delete + re-index** — when a document changes, delete its old embeddings (by document ID metadata filter) and re-index the new version
2. **Incremental indexing** — only re-index changed/new documents, not the entire corpus
3. **Document versioning** — keep version metadata with each chunk; queries can filter by version or date range
4. **Change detection** — webhooks or polling to detect source document changes → trigger re-indexing
5. **TTL (Time to Live)** — auto-expire chunks older than N days for highly dynamic content
6. **Dual index** — keep a "stable" index + a "recent" index; query both, merge results

Challenge: Large knowledge bases can't be re-indexed in real-time. Batch re-indexing (nightly) is common; for real-time needs, use streaming ingestion pipelines (Kafka → embedder → vector DB).


### Q19. How do you optimise RAG for latency in production?
**A:**
1. **Embed caching** — cache query embeddings; many users ask similar questions
2. **Response caching** — full response cache for identical/semantically-similar queries (semantic cache)
3. **Async retrieval** — start retrieval while streaming a "thinking..." message
4. **ANN index tuning** — HNSW ef_search parameter trades recall vs speed; tune for your latency budget
5. **Quantised embeddings** — INT8 embeddings → faster similarity computation
6. **Smaller embedding model** — 384-dim MiniLM is 5× faster than 1536-dim OpenAI embeddings
7. **Pre-filter with metadata** — reduce search space before vector search
8. **Batched retrieval** — if processing multiple queries, batch embed them
9. **Reduce k** — top-3 is faster and often as good as top-20

🏷️ *Nyaya-Pro: Gemini prompt caching for system prompt + query embedding cache → median latency under 800ms including retrieval and generation.*


### Q20. What is the role of metadata filtering in RAG systems?
**A:** Metadata filtering restricts vector search to only chunks matching specific attributes — effectively narrowing the search space before (or during) similarity computation.

Example metadata: `{"source": "BNS", "section_number": "302", "date_updated": "2024-01-01", "document_type": "criminal_law"}`

Query: "What is the punishment for murder?" + filter: `{"source": "BNS"}` → searches only BNS documents.

Benefits:
- Faster search (smaller candidate set)
- More precise results (eliminates irrelevant domains)
- Enables per-user or per-tenant access control
- Enables date-filtered retrieval ("only show docs updated after 2024")

Supported by: Pinecone, Qdrant, Weaviate, Chroma (all support metadata filtering with their vector search).


### Q21. Compare RAG vs fine-tuning. When would you use each?
**A:**
| | RAG | Fine-tuning |
|--|-----|------------|
| **Best for** | Dynamic knowledge, facts, documents | Style, format, domain behaviour |
| **Cost** | Low — add documents, no training | High — GPU hours, data prep |
| **Knowledge updates** | Easy — just re-index | Hard — need to retrain |
| **Cites sources** | Yes | No |
| **Hallucination risk** | Lower (grounded) | Higher (relies on weights) |
| **Latency overhead** | +50-200ms retrieval | None |

**Use RAG when:** Knowledge base updates frequently, you need citations, you need to keep specific facts accurate.
**Use fine-tuning when:** You need consistent style/tone, very specific output format, behaviour change that can't be prompted, or high-volume task where a smaller fine-tuned model is cheaper than a prompted large one.
**Best combination:** RAG + fine-tuned model (fine-tune for format/behaviour, RAG for knowledge grounding).


### Q22. What is query transformation in RAG (HyDE, query decomposition, step-back prompting)?
**A:**
**HyDE (Hypothetical Document Embeddings):** LLM generates a hypothetical ideal answer → embed that answer → search for similar documents. Answer-to-answer matching is better than question-to-answer.

**Query Decomposition:** Break complex questions into sub-questions:
"Compare bail under BNS vs old IPC for murder" → ["What are bail provisions under BNS for murder?", "What were bail provisions under IPC Section 437 for murder?"]

**Step-back prompting:** Generalise the query before searching:
"Who wrote Section 302 BNS?" → "What is the legislative history of India's criminal law?"

**Multi-query:** Generate N paraphrases of the original query, retrieve for all N, deduplicate and merge results.

When to use: When initial retrieval quality is low, for complex multi-part questions, for domain-specific queries where vocabulary mismatch hurts retrieval.


### Q23. How do you implement citation and source attribution in RAG?
**A:**
1. **Chunk-level metadata** — every chunk stores: `{source_file, section_name, page_number, url, date}`
2. **Enforce citation in prompt:** "After your answer, list the exact source sections you used in this format: [Source: BNS Section 302]"
3. **Map citations to retrieved chunks** — after generation, map each citation to its chunk's metadata for verification
4. **Post-processing extraction** — parse the LLM output to extract citation markers and verify they match retrieved chunk metadata
5. **Inline citations** — use a format where citations appear inline: "Murder is defined as... [BNS §101] with punishment of death or life imprisonment [BNS §103]"
6. **Faithfulness verification** — for each cited section, run NLI to confirm the statement is actually supported by that source

🏷️ *Nyaya-Pro: Every response includes a "Sources" field in the JSON response containing the specific legal sections. The frontend displays these as clickable citations.*


### Q24. How do you scale a RAG system to millions of documents?
**A:**
1. **Use a managed vector DB** — Pinecone, Weaviate, or Qdrant handle millions of vectors; don't use FAISS in-memory at this scale
2. **ANN indexing** — HNSW or IVF (Inverted File Index) for sub-linear search time
3. **Embedding quantisation** — INT8 or binary quantisation to reduce memory footprint
4. **Sharding** — split the vector DB across multiple nodes; route queries to relevant shards based on metadata
5. **Pre-filtering with metadata** — dramatically reduce the search space before vector similarity
6. **Async ingestion pipeline** — Kafka/Celery queue → embedding workers → vector DB (don't block on indexing)
7. **Batch embedding** — embed documents in batches of 64-512 for GPU efficiency
8. **Distributed embedding** — parallel embedding workers for large corpus ingestion
9. **Horizontal scaling** — scale vector DB read replicas for query throughput


### Q25. What is parent-child chunking, and how does it improve retrieval?
**A:** Parent-child chunking decouples the retrieval unit from the synthesis unit:

- **Child chunks** (small, e.g., 128 tokens): Used for vector search. Small = precise semantic matching.
- **Parent chunks** (large, e.g., 512-1024 tokens): Returned to the LLM. Large = rich context.

When a child chunk matches a query, the system returns its parent chunk to the LLM.

Why better than flat chunking:
- Flat chunking: either precise retrieval OR rich context — can't have both
- Parent-child: precise retrieval via small child + rich context via large parent

Implementation: Each child chunk stores a `parent_id` metadata field. After retrieval, replace child chunks with their parents before building the prompt.


### Q26. Your RAG system is hallucinating despite having the right context. How do you fix it?
**A:** Right context, wrong generation = generation-side problem.

1. **Tighten system prompt:** "Answer ONLY using the information in the provided context. Do not add any information not present in the context."
2. **Require citations** — "Every factual statement must include a reference to the source section."
3. **Temperature = 0** — eliminates creative drift
4. **Reduce chunk count** — 10 chunks = noisy context; 3 highly-reranked chunks = focused context
5. **Faithfulness checker** — second LLM call: "Is every claim in this answer supported by this context? List any unsupported claims." If any found, regenerate.
6. **NLI verification** — use a natural language inference model to check each generated claim against the source

🏷️ *Nyaya-Pro: cross-encoder reranking cuts to top-3 + system prompt strictly limits generation to context + mandatory section citation = near-zero hallucination in production.*


### Q27. Your RAG chunk overlap causes redundant results. How do you reduce redundancy?
**A:**
1. **Maximum Marginal Relevance (MMR)** — retrieval algorithm that balances relevance AND diversity; penalises chunks too similar to already-selected chunks. LangChain supports `search_type="mmr"`.
2. **Deduplication post-retrieval** — after retrieving top-k, remove near-duplicate chunks (cosine similarity > 0.95 with an already-selected chunk)
3. **Parent-chunk deduplication** — if two child chunks belong to the same parent, return the parent once
4. **Reduce overlap at indexing time** — smaller overlap (e.g., 20 tokens vs 100 tokens) reduces redundancy but risks boundary information loss
5. **Deduplicate by source ID** — if two chunks from the same source section are returned, keep only the highest-scoring one


### Q28. Your RAG retrieval is too slow. How do you speed it up?
**A:**
1. **Switch from FAISS flat to HNSW** — O(log N) ANN search vs O(N) exact search
2. **Pre-filter with metadata** — narrow the search space before vector similarity computation
3. **Quantise vectors** — INT8 or binary quantisation → faster distance computation
4. **Reduce embedding dimension** — use 384-dim instead of 1536-dim model
5. **Use a managed cloud vector DB** — Pinecone/Qdrant have optimised C++ index implementations
6. **Cache query embeddings** — if the same/similar query comes again, reuse the embedding
7. **Reduce k** — searching for top-3 is faster than top-20
8. **Async retrieval** — run retrieval in a background thread/task while streaming an acknowledgement to the user


### Q29. Your RAG system returns duplicate results. How do you deduplicate?
**A:**
1. **MMR (Maximum Marginal Relevance)** — use at retrieval time; actively selects diverse results
2. **Post-retrieval deduplication** — compare all retrieved chunks pairwise; if cosine similarity > threshold (0.92), keep only the highest-scoring one
3. **Source deduplication** — group chunks by source document; return at most N chunks per source
4. **Content hashing** — compute MD5/SHA of chunk text; deduplicate by hash before inserting into vector DB (prevents duplicate indexing)
5. **Parent ID merging** — if multiple child chunks share a parent, return the parent once
6. **Sliding window with smaller step** — if overlap was too large during chunking, re-chunk with a larger step


### Q30. Your RAG system needs per-user access control. How do you implement it?
**A:**
1. **Namespace per user/tenant** — store each user's documents in a separate namespace/collection in the vector DB (Pinecone namespaces, Chroma collections)
2. **Metadata filtering** — tag every chunk with `{"user_id": "123", "tenant_id": "acme"}` and filter at query time
3. **Row-level security** — the retriever passes the user's access metadata as a filter; the vector DB only searches their allowed chunks
4. **JWT-based retrieval** — decode user's JWT token → extract allowed document IDs → pass as metadata filter
5. **Encryption** — encrypt embeddings per tenant so a DB compromise doesn't expose cross-tenant data
6. **Audit logging** — log every retrieval with user_id, query, and returned document IDs for compliance

🏷️ *Nyaya-Pro has a simpler version: user case history is namespaced by user_id. Each user's private notes are only retrievable with their authenticated session.*


### Q31. Your RAG system fails on domain-specific jargon. How do you fix it?
**A:**
1. **Domain-specific embedding model** — fine-tune or use an embedding model trained on domain text (legal-bert, bioBERT for medical, FinBERT for finance)
2. **Glossary injection** — add a glossary of domain terms and their definitions to the system prompt
3. **Query expansion** — expand abbreviations and jargon in the query before embedding: "BNSS" → "Bharatiya Nagarik Suraksha Sanhita 2023"
4. **Hybrid search** — BM25 exact keyword match catches jargon that semantic search misses
5. **Specialised tokeniser** — retrain the tokeniser with domain vocabulary so terms don't get fragmented into meaningless sub-words
6. **More domain data for retrieval** — add domain-specific documents (textbooks, glossaries) to the knowledge base

🏷️ *Nyaya-Pro: I preprocess all queries to expand legal abbreviations before embedding. I also add a legal glossary to the FAISS index.*


### Q32. Your text-only RAG system needs to handle images and tables. How do you extend it?
**A:**
**For images:**
1. Use a vision-language model (GPT-4V, Gemini) to describe images → store descriptions as text chunks
2. Use multi-modal embeddings (CLIP) to embed images directly; search with text query

**For tables:**
1. Extract tables with pdfplumber/Camelot → convert to markdown or CSV → embed as text
2. Serialise table as text: "Header1 | Header2 | Header3
 val1 | val2 | val3"
3. Use NL2SQL for structured table queries if data is in a database

**For PDFs with mixed content:**
1. Azure Document Intelligence / AWS Textract — handles layout, tables, and images in one pass
2. Unstructured.io — open-source multimodal document parser
3. Send full page images to a vision LLM (Gemini 2.5 Flash) — it processes text + tables + diagrams together

🏷️ *ExamGenie AI: PDFs with diagrams → send full page images to Gemini 2.5 Flash — it reads both text and visual content to generate contextually accurate MCQs.*


### Q33. Your RAG knowledge base gets updated frequently and needs versioning. How do you manage it?
**A:**
1. **Chunk-level version metadata** — tag each chunk with `{"version": "2.1", "effective_date": "2024-06-01"}`
2. **Document-level soft delete** — when a document is updated, mark old chunks as `{"active": false}` and insert new chunks with new version
3. **Version-filtered retrieval** — query filter: `{"active": true}` to always get current versions
4. **Snapshot indexes** — maintain a "point-in-time" snapshot index for audit purposes (retrieve what the system knew on date X)
5. **Blue-green index** — build new index while old one serves traffic; switch atomically when new index is ready
6. **Change log** — log every index change: what document, what version, who triggered it, when

For compliance-sensitive systems: retain old versions for audit trails; never truly delete.


### Q34. Your RAG system fails on multi-hop questions. How do you fix it?
**A:** Multi-hop: "What is the punishment for abetment of murder under BNS and how does it compare to abetment under the old IPC?"

Fixes:
1. **Query decomposition** — automatically break into sub-questions; retrieve and answer each; synthesise
2. **Iterative retrieval** — retrieve → generate partial answer → use partial answer to formulate next retrieval query → repeat
3. **GraphRAG** — build a knowledge graph; traverse entity relationships to gather connected facts
4. **Multi-vector retrieval** — retrieve with multiple query reformulations simultaneously; merge results
5. **Chain-of-thought retrieval** — CoT prompt that explicitly plans "what do I need to look up first, second, third?"
6. **Increase k** — retrieve more chunks (top-20 instead of top-5) to capture all relevant pieces; use reranking to clean up

🏷️ *Nyaya-Pro handles cross-act references (BNS Section on abetment references BNSS for procedure) by cross-indexing both acts in the same FAISS store with source metadata.*


### Q35. Your enterprise RAG returns contradictory answers from different source documents. How do you resolve conflicts?
**A:**
1. **Surface the conflict to the user** — "Note: Document A says X while Document B says Y. Document B is more recent (2024 vs 2022)."
2. **Recency-based resolution** — prefer the most recently updated document (metadata date filter)
3. **Authority-based resolution** — rank sources by authority level (official legislation > academic commentary > news articles)
4. **Confidence scoring** — score each retrieved answer's confidence; present the highest-confidence one, note the conflict
5. **Provenance tracking** — show which source supports which claim so the user can judge
6. **Consensus detection** — if 3 of 4 retrieved sources agree, present the majority view and note the dissenter
7. **Expert routing** — for high-stakes conflicts, route to human review


### Q36. Your RAG system returns outdated answers. How do you keep it current?
**A:**
1. **Scheduled re-indexing** — nightly/weekly re-index updated documents; fastest pragmatic fix
2. **Metadata date filtering** — add `updated_date` to chunks; query filter to prefer recent documents
3. **Real-time ingestion pipeline** — webhook from source system → embedding worker → update vector DB immediately on document change
4. **TTL on chunks** — auto-expire highly volatile data (news, prices) after N hours
5. **Hybrid RAG + live data** — for real-time data (stock prices, current news), don't use RAG; use tool calls to live APIs
6. **Freshness score** — incorporate document age into retrieval scoring: freshness_boost = exp(-days_old/90)
7. **User feedback loop** — flag "outdated answer" option; trigger manual review and re-indexing


### Q37. Your RAG system struggles with PDF documents containing tables and layouts. How do you fix PDF parsing?
**A:** Standard PDF parsers (PyPDF2) extract text linearly, destroying table structure.

Fixes:
1. **pdfplumber** — extracts text with bounding box info; can reconstruct tables using `.extract_tables()`
2. **Camelot** — dedicated table extraction from PDFs; works well for digitally-created PDFs
3. **Azure Document Intelligence** — cloud API that handles complex layouts, multi-column, tables, forms — best quality
4. **AWS Textract** — similar to Azure DI; great for scanned PDFs too
5. **Unstructured.io** — open-source, handles mixed content (text + tables + images) in one pass
6. **Page image approach** — render PDF page as image → send to Gemini/GPT-4V → get structured extraction with layout preserved
7. **PyMuPDF (fitz)** — better than PyPDF2 for complex layouts; preserves text positioning

🏷️ *ExamGenie AI: for study PDFs with formulas and diagrams, I render pages as images and send to Gemini — zero structure loss.*
